In [ ]:
!pip install -q ultralytics pyyaml pandas pillow

In [ ]:
from pathlib import Path
from collections import Counter, defaultdict
import os
import re
import random
import shutil
import yaml
import pandas as pd
import numpy as np

import torch
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Không có GPU. Hãy bật Accelerator = GPU trong Kaggle Notebook Settings.")

## 1. Cấu hình chung

`RUN_LEVEL`:
- `"quick"`: chạy ít cấu hình để test pipeline.
- `"recommended"`: chạy baseline + 2 cấu hình đáng thử nhất.
- `"full"`: chạy ablation đầy đủ hơn.

Mặc định đang để `"full"` để làm đúng kiểu ablation.

In [ ]:
# Có thể đổi thành "quick" hoặc "recommended" nếu muốn chạy ít hơn.
RUN_LEVEL = "custom"

# Nếu đã train rồi và muốn dùng lại checkpoint cũ thì để False.
# Nếu muốn train lại từ đầu toàn bộ thì để True.
FORCE_RETRAIN = False

# Có chạy predict ảnh test bằng model tốt nhất không.
RUN_PREDICT_EXAMPLES = True

# Thư mục output
WORK_DIR = Path("/kaggle/working")
PROJECT_DIR = WORK_DIR / "runs" / "rice_pest_ablation"
RESULTS_DIR = WORK_DIR / "ablation_results"
CHECKPOINT_DIR = RESULTS_DIR / "checkpoints"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Train settings chung
DEFAULT_EPOCHS = 120
DEFAULT_BATCH = -1
DEFAULT_PATIENCE = 20
DEFAULT_WORKERS = 2
DEVICE = 0 if torch.cuda.is_available() else "cpu"

## 2. Tự tìm dataset YOLO trong `/kaggle/input`

In [ ]:
def find_yolo_dataset_root(search_root="/kaggle/input"):
    search_root = Path(search_root)
    yaml_files = list(search_root.rglob("data.yaml"))
    print("Found data.yaml files:", len(yaml_files))

    for yaml_path in yaml_files:
        root = yaml_path.parent

        has_train = (root / "train" / "images").exists() and (root / "train" / "labels").exists()
        has_valid = (root / "valid" / "images").exists() and (root / "valid" / "labels").exists()
        has_test = (root / "test" / "images").exists() and (root / "test" / "labels").exists()

        if has_train and has_valid and has_test:
            return root, yaml_path

    raise FileNotFoundError(
        "Không tìm thấy dataset YOLO hợp lệ. "
        "Hãy kiểm tra /kaggle/input có data.yaml, train/images, valid/images, test/images không."
    )

DATASET_DIR, original_yaml_path = find_yolo_dataset_root("/kaggle/input")

print("DATASET_DIR:", DATASET_DIR)
print("original_yaml_path:", original_yaml_path)

print("\nRoot files:")
print(os.listdir(DATASET_DIR)[:30])

## 3. Patch `data.yaml`

Vì `/kaggle/input` là read-only và đường dẫn trong `data.yaml` có thể là relative path kiểu Roboflow, ta tạo file mới ở `/kaggle/working/data_patched.yaml`.

In [ ]:
with open(original_yaml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

data["path"] = str(DATASET_DIR.resolve())
data["train"] = "train/images"
data["val"] = "valid/images"
data["test"] = "test/images"

PATCHED_YAML_PATH = WORK_DIR / "data_patched.yaml"

with open(PATCHED_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print("Patched YAML:", PATCHED_YAML_PATH)
print("\nNội dung data_patched.yaml:")
print(PATCHED_YAML_PATH.read_text())

CLASS_NAMES = data["names"]
NC = data["nc"]

print("\nClasses:")
for i, name in enumerate(CLASS_NAMES):
    print(i, name)

## 4. Kiểm tra dataset: số ảnh, số bbox, label lỗi, phân bố class

In [ ]:
def image_files(split):
    img_dir = DATASET_DIR / split / "images"
    return [p for p in img_dir.glob("*") if p.suffix.lower() in [".jpg", ".jpeg", ".png"]]

def label_files(split):
    label_dir = DATASET_DIR / split / "labels"
    return list(label_dir.glob("*.txt"))

def inspect_yolo_labels():
    rows = []
    bad_rows = []
    empty_label_files = []
    split_class_counts = defaultdict(Counter)
    total_class_counts = Counter()

    for split in ["train", "valid", "test"]:
        imgs = image_files(split)
        labels = label_files(split)

        for label_path in labels:
            text = label_path.read_text(encoding="utf-8").strip()
            if not text:
                empty_label_files.append(str(label_path))
                continue

            for line_no, line in enumerate(text.splitlines(), start=1):
                parts = line.split()
                if len(parts) != 5:
                    bad_rows.append((str(label_path), line_no, line, "len != 5"))
                    continue

                try:
                    cls, x, y, w, h = map(float, parts)
                except Exception as e:
                    bad_rows.append((str(label_path), line_no, line, f"parse error: {e}"))
                    continue

                ok = (0 <= cls < NC and 0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1)
                if not ok:
                    bad_rows.append((str(label_path), line_no, line, "value out of range"))
                    continue

                cls_id = int(cls)
                split_class_counts[split][cls_id] += 1
                total_class_counts[cls_id] += 1

        rows.append({
            "split": split,
            "images": len(imgs),
            "labels": len(labels),
            "bbox": sum(split_class_counts[split].values()),
        })

    df_split = pd.DataFrame(rows)

    class_rows = []
    for cls_id, name in enumerate(CLASS_NAMES):
        class_rows.append({
            "class_id": cls_id,
            "class_name": name,
            "train_bbox": split_class_counts["train"][cls_id],
            "valid_bbox": split_class_counts["valid"][cls_id],
            "test_bbox": split_class_counts["test"][cls_id],
            "total_bbox": total_class_counts[cls_id],
        })
    df_class = pd.DataFrame(class_rows)

    return df_split, df_class, bad_rows, empty_label_files

df_split, df_class, bad_rows, empty_label_files = inspect_yolo_labels()

print("Split summary:")
display(df_split)

print("\nClass bbox distribution:")
display(df_class)

print("\nEmpty label files:", len(empty_label_files))
print("Bad label rows:", len(bad_rows))

if bad_rows:
    print("Một vài label lỗi:")
    for item in bad_rows[:20]:
        print(item)

## 5. Kiểm tra leakage theo base filename

Với dataset Roboflow, tên file thường có dạng:

`original_name_jpg.rf.hash.jpg`

Ta lấy phần `original_name` để kiểm tra cùng một ảnh gốc có xuất hiện ở nhiều split không.

In [ ]:
pattern = re.compile(r"(.+?)_(jpg|jpeg|png)\.rf\.[^.]+\.(jpg|jpeg|png)$", re.IGNORECASE)

def get_base_name(img_path):
    m = pattern.match(img_path.name)
    return m.group(1) if m else img_path.stem

base_to_splits = defaultdict(set)

for split in ["train", "valid", "test"]:
    for img_path in image_files(split):
        base_to_splits[get_base_name(img_path)].add(split)

cross_split = {base: splits for base, splits in base_to_splits.items() if len(splits) > 1}

print("Số base filename xuất hiện ở nhiều split:", len(cross_split))

if cross_split:
    print("Một vài ví dụ có nguy cơ leakage:")
    for base, splits in list(cross_split.items())[:30]:
        print(base, sorted(splits))
else:
    print("OK: chưa thấy trùng base filename giữa train/valid/test.")

## 6. Định nghĩa các cấu hình ablation

Notebook này chỉ chạy nhóm **YOLO11**.

Các experiment sẽ chạy:

Tất cả experiment được train với `epochs=120` và `patience=15`.

Không thêm augmentation riêng vì dataset đã được Roboflow augment trước đó.

In [ ]:
ALL_EXPERIMENTS = [
    {
        "name": "E01_yolov8n_img320_default",
        "model": "yolov8n.pt",
        "family": "YOLOv8",
        "size": "n",
        "imgsz": 320,
        "epochs": 100,
        "patience": 15,
        "description": "Baseline: YOLOv8n, imgsz=320, default augmentation",
        "train_overrides": {},
    },
    {
        "name": "E02_yolov8n_img416_default",
        "model": "yolov8n.pt",
        "family": "YOLOv8",
        "size": "n",
        "imgsz": 416,
        "epochs": 120,
        "patience": 15,
        "description": "Image size ablation: YOLOv8n, imgsz=416",
        "train_overrides": {},
    },
    {
        "name": "E03_yolov8s_img320_default",
        "model": "yolov8s.pt",
        "family": "YOLOv8",
        "size": "s",
        "imgsz": 320,
        "epochs": 120,
        "patience": 15,
        "description": "Model size ablation: YOLOv8s, imgsz=320",
        "train_overrides": {},
    },
    {
        "name": "E04_yolov8s_img416_default",
        "model": "yolov8s.pt",
        "family": "YOLOv8",
        "size": "s",
        "imgsz": 416,
        "epochs": 120,
        "patience": 15,
        "description": "YOLOv8 strong config: YOLOv8s, imgsz=416",
        "train_overrides": {},
    },
    {
        "name": "E05_yolo26n_img416_default",
        "model": "yolo26n.pt",
        "family": "YOLO26",
        "size": "n",
        "imgsz": 416,
        "epochs": 120,
        "patience": 15,
        "description": "New generation candidate: YOLO26n, imgsz=416",
        "train_overrides": {},
    },
    {
        "name": "E06_yolo26s_img416_default",
        "model": "yolo26s.pt",
        "family": "YOLO26",
        "size": "s",
        "imgsz": 416,
        "epochs": 120,
        "patience": 15,
        "description": "New generation candidate: YOLO26s, imgsz=416",
        "train_overrides": {},
    },
    {
        "name": "E07_yolo11n_img416_default",
        "model": "yolo11n.pt",
        "family": "YOLO11",
        "size": "n",
        "imgsz": 416,
        "epochs": 120,
        "patience": 15,
        "description": "App candidate: YOLO11n, imgsz=416",
        "train_overrides": {},
    },
    {
        "name": "E08_yolo11s_img416_default",
        "model": "yolo11s.pt",
        "family": "YOLO11",
        "size": "s",
        "imgsz": 416,
        "epochs": 120,
        "patience": 15,
        "description": "App candidate: YOLO11s, imgsz=416",
        "train_overrides": {},
    },
]

selected_names = [
    "E07_yolo11n_img416_default",
    "E08_yolo11s_img416_default"
]

EXPERIMENTS = [exp for exp in ALL_EXPERIMENTS if exp["name"] in selected_names]

pd.DataFrame([
    {
        "name": e["name"],
        "family": e["family"],
        "size": e["size"],
        "model": e["model"],
        "imgsz": e["imgsz"],
        "epochs": e["epochs"],
        "patience": e["patience"],
        "description": e["description"],
    }
    for e in EXPERIMENTS
])

## 6.1. Vì sao so YOLOv8, YOLO11 và YOLO26?

Notebook này giữ `YOLOv8n/YOLOv8s` làm baseline vì đây là dòng phổ biến, dễ giải thích và dễ bảo vệ trong báo cáo.

Sau đó thêm `YOLO11n/YOLO11s` vì YOLO11 là lựa chọn trung gian rất phù hợp cho ứng dụng: mới hơn YOLOv8, ổn định, nhẹ và dễ deploy.

Cuối cùng thêm `YOLO26n/YOLO26s` như nhóm model thế hệ mới hơn, phù hợp nếu muốn tối ưu app/edge deployment.

Cách so chính:

- `YOLOv8n 416` vs `YOLO11n 416` vs `YOLO26n 416`: so cùng bản nano.
- `YOLOv8s 416` vs `YOLO11s 416` vs `YOLO26s 416`: so cùng bản small.
- `YOLOv8n 320` vs `YOLOv8n 416`: xem ảnh lớn hơn có cải thiện không.
- `YOLOv8n 320` vs `YOLOv8s 320`: xem model lớn hơn có cải thiện không.

Không chạy `m/l/x` mặc định vì các bản đó nặng hơn nhiều, train lâu hơn và có thể không cần thiết cho app demo.

## 7. Helper functions: train, validate, tính F1, lưu kết quả

In [ ]:
def safe_float(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def calc_f1(p, r):
    p, r = safe_float(p), safe_float(r)
    if np.isnan(p) or np.isnan(r) or (p + r) == 0:
        return np.nan
    return 2 * p * r / (p + r)

def get_results_dict(metrics_obj):
    d = getattr(metrics_obj, "results_dict", {})
    return {
        "precision": safe_float(d.get("metrics/precision(B)", np.nan)),
        "recall": safe_float(d.get("metrics/recall(B)", np.nan)),
        "mAP50": safe_float(d.get("metrics/mAP50(B)", np.nan)),
        "mAP50-95": safe_float(d.get("metrics/mAP50-95(B)", np.nan)),
        "fitness": safe_float(d.get("fitness", np.nan)),
    }

def extract_per_class_metrics(metrics_obj, exp_name, split):
    rows = []
    box = getattr(metrics_obj, "box", None)

    if box is None:
        return rows

    ap_class_index = getattr(metrics_obj, "ap_class_index", None)
    if ap_class_index is None:
        ap_class_index = getattr(box, "ap_class_index", None)

    if ap_class_index is None:
        return rows

    # Các thuộc tính này có thể thay đổi nhẹ theo version Ultralytics,
    # nên dùng getattr và fallback an toàn.
    p_arr = np.array(getattr(box, "p", []), dtype=float)
    r_arr = np.array(getattr(box, "r", []), dtype=float)
    f1_arr = np.array(getattr(box, "f1", []), dtype=float)
    ap50_arr = np.array(getattr(box, "ap50", []), dtype=float)
    ap_arr = np.array(getattr(box, "ap", []), dtype=float)

    nt_per_class = np.array(getattr(metrics_obj, "nt_per_class", []))
    nt_per_image = np.array(getattr(metrics_obj, "nt_per_image", []))

    for i, cls_id in enumerate(ap_class_index):
        cls_id = int(cls_id)
        p = p_arr[i] if i < len(p_arr) else np.nan
        r = r_arr[i] if i < len(r_arr) else np.nan
        f1 = f1_arr[i] if i < len(f1_arr) else calc_f1(p, r)
        ap50 = ap50_arr[i] if i < len(ap50_arr) else np.nan
        ap = ap_arr[i] if i < len(ap_arr) else np.nan

        rows.append({
            "experiment": exp_name,
            "split": split,
            "class_id": cls_id,
            "class_name": CLASS_NAMES[cls_id],
            "images": int(nt_per_image[i]) if i < len(nt_per_image) else np.nan,
            "instances": int(nt_per_class[i]) if i < len(nt_per_class) else np.nan,
            "precision": p,
            "recall": r,
            "f1": f1,
            "mAP50": ap50,
            "mAP50-95": ap,
        })

    return rows

def validate_model(weight_path, exp_name, imgsz, split):
    model = YOLO(str(weight_path))
    metrics = model.val(
        data=str(PATCHED_YAML_PATH),
        split=split,
        imgsz=imgsz,
        batch=16,
        device=DEVICE,
        plots=True,
    )

    overall = get_results_dict(metrics)
    overall["f1"] = calc_f1(overall["precision"], overall["recall"])
    overall.update({
        "experiment": exp_name,
        "split": split,
        "weight_path": str(weight_path),
        "imgsz": imgsz,
    })

    per_class_rows = extract_per_class_metrics(metrics, exp_name, split)
    return overall, per_class_rows

def train_one_experiment(exp):
    exp_name = exp["name"]
    run_dir = PROJECT_DIR / exp_name
    best_pt = run_dir / "weights" / "best.pt"
    last_pt = run_dir / "weights" / "last.pt"

    print("=" * 100)
    print("Experiment:", exp_name)
    print("Description:", exp["description"])
    print("Model:", exp["model"], "| imgsz:", exp["imgsz"])
    print("=" * 100)

    if best_pt.exists() and not FORCE_RETRAIN:
        print("Found existing best.pt, skip training:", best_pt)
    else:
        model = YOLO(exp["model"])

        train_args = {
            "data": str(PATCHED_YAML_PATH),
            "task": "detect",
            "imgsz": exp["imgsz"],
            "epochs": exp.get("epochs", DEFAULT_EPOCHS),
            "batch": DEFAULT_BATCH,
            "patience": exp.get("patience", DEFAULT_PATIENCE),
            "device": DEVICE,
            "workers": DEFAULT_WORKERS,
            "optimizer": "auto",
            "pretrained": True,
            "seed": SEED,
            "deterministic": True,
            "project": str(PROJECT_DIR),
            "name": exp_name,
            "exist_ok": True,
            "save": True,
            "save_period": -1,
            "plots": True,
            "val": True,
            "close_mosaic": 10,
        }

        train_args.update(exp.get("train_overrides", {}))

        results = model.train(**train_args)
        print("Train done.")
        print("Save dir:", results.save_dir)

        best_pt = Path(results.save_dir) / "weights" / "best.pt"
        last_pt = Path(results.save_dir) / "weights" / "last.pt"

    if not best_pt.exists():
        raise FileNotFoundError(f"Không tìm thấy best.pt cho {exp_name}: {best_pt}")

    copied_best = CHECKPOINT_DIR / f"{exp_name}_best.pt"
    shutil.copy2(best_pt, copied_best)

    print("Best checkpoint:", best_pt)
    print("Copied to:", copied_best)

    return copied_best, last_pt

## 8. Chạy ablation

Cell này sẽ train từng experiment, validate trên `val` và `test`, rồi gom bảng kết quả.

In [ ]:
summary_rows = []
per_class_rows_all = []

for exp in EXPERIMENTS:
    best_weight, last_weight = train_one_experiment(exp)

    for split in ["val", "test"]:
        print("\nValidating:", exp["name"], "| split:", split)
        overall, per_class_rows = validate_model(
            weight_path=best_weight,
            exp_name=exp["name"],
            imgsz=exp["imgsz"],
            split=split,
        )

        overall.update({
            "family": exp.get("family", ""),
            "size": exp.get("size", ""),
            "model": exp["model"],
            "epochs": exp["epochs"],
            "patience": exp["patience"],
            "description": exp["description"],
        })

        summary_rows.append(overall)
        per_class_rows_all.extend(per_class_rows)

df_summary = pd.DataFrame(summary_rows)
df_per_class = pd.DataFrame(per_class_rows_all)

summary_csv = RESULTS_DIR / "ablation_summary.csv"
per_class_csv = RESULTS_DIR / "per_class_metrics.csv"

df_summary.to_csv(summary_csv, index=False)
df_per_class.to_csv(per_class_csv, index=False)

print("Saved summary:", summary_csv)
print("Saved per-class metrics:", per_class_csv)

display(
    df_summary.sort_values(["split", "mAP50-95"], ascending=[True, False])
    [["split", "experiment", "family", "size", "model", "imgsz", "precision", "recall", "f1", "mAP50", "mAP50-95", "fitness"]]
    .round(4)
)

## 9. Bảng so sánh riêng trên test

Ưu tiên chọn model theo `mAP50-95` trên test. Nếu app cần ít bỏ sót hơn thì xem thêm `recall` và `f1`.

In [ ]:
df_test = df_summary[df_summary["split"] == "test"].copy()
df_test = df_test.sort_values("mAP50-95", ascending=False)

display(
    df_test[["experiment", "family", "size", "model", "imgsz", "precision", "recall", "f1", "mAP50", "mAP50-95", "weight_path"]]
    .round(4)
)

best_row = df_test.iloc[0]
best_overall_src = Path(best_row["weight_path"])
best_overall_dst = CHECKPOINT_DIR / "best_overall.pt"
shutil.copy2(best_overall_src, best_overall_dst)

print("Best experiment by test mAP50-95:", best_row["experiment"])
print("Best test mAP50-95:", round(best_row["mAP50-95"], 4))
print("Best test F1:", round(best_row["f1"], 4))
print("Best overall checkpoint:", best_overall_dst)

## 9.1. Bảng so sánh riêng nhóm YOLO11

Cell này tạo bảng riêng cho các experiment trong notebook này.


In [ ]:
df_family_compare = df_test.copy()

display(
    df_family_compare[
        ["family", "size", "experiment", "precision", "recall", "f1", "mAP50", "mAP50-95"]
    ]
    .sort_values(["size", "mAP50-95"], ascending=[True, False])
    .round(4)
)

family_compare_csv = RESULTS_DIR / "family_compare_yolo11.csv"
df_family_compare.to_csv(family_compare_csv, index=False)
print("Saved:", family_compare_csv)


## 10. Bảng per-class của model tốt nhất trên test

Bảng này giúp xem lớp nào yếu để bàn luận trong báo cáo.

In [ ]:
best_exp = best_row["experiment"]

df_best_per_class_test = df_per_class[
    (df_per_class["experiment"] == best_exp) &
    (df_per_class["split"] == "test")
].copy()

df_best_per_class_test = df_best_per_class_test.sort_values("mAP50-95", ascending=True)

display(
    df_best_per_class_test[
        ["class_id", "class_name", "images", "instances", "precision", "recall", "f1", "mAP50", "mAP50-95"]
    ].round(4)
)

weak_classes_csv = RESULTS_DIR / "best_model_test_per_class.csv"
df_best_per_class_test.to_csv(weak_classes_csv, index=False)
print("Saved:", weak_classes_csv)

## 11. Predict thử ảnh test bằng model tốt nhất

Kết quả ảnh có bounding box sẽ nằm trong thư mục `runs/rice_pest_ablation/predict_best_test_examples`.

In [ ]:
if RUN_PREDICT_EXAMPLES:
    best_model = YOLO(str(best_overall_dst))
    test_img_dir = DATASET_DIR / "test" / "images"

    pred_results = best_model.predict(
        source=str(test_img_dir),
        imgsz=int(best_row["imgsz"]),
        conf=0.25,
        iou=0.7,
        save=True,
        project=str(PROJECT_DIR),
        name="predict_best_test_examples",
        max_det=20,
    )

    print("Ảnh dự đoán được lưu tại:")
    print(PROJECT_DIR / "predict_best_test_examples")
else:
    print("Skipped prediction examples.")

## 12. Nếu muốn tăng Recall khi đưa vào app

Cell này chỉ dùng cho inference/app, không thay đổi chất lượng train.  
Giảm `conf` giúp bớt bỏ sót, nhưng có thể tăng false positive.

In [ ]:
# Ví dụ predict với confidence thấp hơn để giảm bỏ sót.
# Chỉ chạy khi cần xem app/demo.

RUN_LOW_CONF_PREDICT = False

if RUN_LOW_CONF_PREDICT:
    best_model = YOLO(str(best_overall_dst))
    test_img_dir = DATASET_DIR / "test" / "images"

    pred_low_conf = best_model.predict(
        source=str(test_img_dir),
        imgsz=int(best_row["imgsz"]),
        conf=0.15,
        iou=0.7,
        save=True,
        project=str(PROJECT_DIR),
        name="predict_best_test_conf015",
        max_det=20,
    )

    print("Low-conf prediction saved to:")
    print(PROJECT_DIR / "predict_best_test_conf015")

## 13. Nén toàn bộ kết quả để tải về

Sau khi chạy xong, file zip nằm ở:

`/kaggle/working/rice_pest_yolo11_ablation_outputs.zip`


In [ ]:
zip_path = WORK_DIR / "rice_pest_yolo11_ablation_outputs.zip"

# Nén kết quả quan trọng: CSV + checkpoints + runs.
!cd /kaggle/working && zip -qr rice_pest_yolo11_ablation_outputs.zip ablation_results runs/rice_pest_ablation data_patched.yaml

print("ZIP output:", zip_path)


## Ghi chú diễn giải kết quả

- `Precision`: model dự đoán sâu thì đúng bao nhiêu phần.
- `Recall`: trong số sâu thật, model bắt được bao nhiêu phần.
- `F1`: trung hòa giữa Precision và Recall.
- `mAP50`: chất lượng phát hiện với IoU ngưỡng 0.50.
- `mAP50-95`: metric nghiêm ngặt hơn, thường nên dùng để chọn model tốt nhất.

Với bài app phát hiện sâu bệnh, nên ưu tiên:
1. `mAP50-95` cao.
2. `F1` cao.
3. `Recall` không quá thấp, vì bỏ sót sâu là vấn đề lớn.